In [ ]:
# ============================================================
# SETUP - Correr esta celda primero
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install torch_geometric

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import precision_recall_curve
from torch_geometric.loader import DataLoader


In [ ]:
from utils.datasets      import NF_IDS_Dataset
from utils.models        import SimpleMLP, EdgeGRU_Baseline, StaticGNN_Identity, ST_GNN_Identity
from utils.metrics       import calculate_metrics_gnn
from utils.training      import evaluate
from utils.visualization import plot_comparison, plot_radar_chart


# Functions

## Change threshold

In [11]:
def change_threshold(model_class, model_config,
                     val_loader, experiment_name,
                     device='cpu',
                     results_dir="./results_earlystopping",
                     verbose=False):

    df_metrics = pd.read_csv(f"{results_dir}/logs/{experiment_name}/run_metrics_{experiment_name}.csv")

    all_optimal_thresholds = {}
    all_results_in_memory = []
    for exp_id in range(len(df_metrics)):
        # Setup
        df_exp = df_metrics.iloc[exp_id,:].copy()
        seed = df_exp['seed']

        model_config['model_name'] = df_exp['model_name']

        if verbose:
            print(f"\nChanging threshold for: {model_config['model_name']}")

        model_config['type'] = df_exp['type']
        run_id = df_exp['extra_run_id']

        pos_weight = torch.tensor([model_config['extra_params']['pos_weight']]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

        if verbose:
            print("\n OLD METRICS (MAX RECALL FOR PRECISION=0.9):")
            print(f" Precision: {df_exp['Precision']:.4f} | Recall: {df_exp['Recall']:.4f} | F1: {df_exp['F1']:.4f} | F2: {df_exp['F2']:.4f} | AUC-PR: {df_exp['AUC-PR']:.4f} | AUC-ROC: {df_exp['AUC-ROC']:.4f} | FPR: {df_exp['FPR']:.4f}")

        # Load model
        model_paths = glob.glob(os.path.join(results_dir,"saved_models",experiment_name,f"{run_id}_*.pth"))
        # Ensure we are loading a single file, assuming glob returns a list with one item
        model_path = model_paths[0] if model_paths else None
        if model_path is None:
            print(f"Error: No model found for run_id {run_id} in experiment {experiment_name}")
            continue

        model = model_class(**model_config['model_params']).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))

        # Evaluate
        model.eval()
        _, y_true, y_probs = evaluate(model, val_loader, criterion, device, is_temporal)

        # New threshold (max F1)
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
        precisions = precisions[:-1]
        recalls = recalls[:-1]

        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        new_best_th = thresholds[best_idx]
        all_optimal_thresholds[f"seed_{seed}"] = new_best_th

        # Calculate metrics with new threshold
        new_metrics = calculate_metrics_gnn(y_true, y_probs, prob_threshold=new_best_th)
        new_metrics['optimal_threshold'] = new_best_th

        if verbose:
            print("\n NEW METRICS (MAX F1):")
            print(f" Precision: {new_metrics['Precision']:.4f} | Recall: {new_metrics['Recall']:.4f} | F1: {new_metrics['F1']:.4f} | F2: {new_metrics['F2']:.4f} | AUC-PR: {new_metrics['AUC-PR']:.4f} | AUC-ROC: {new_metrics['AUC-ROC']:.4f} | FPR: {new_metrics['FPR']:.4f}\n")


        # Save
        df_new_row = pd.DataFrame([new_metrics]) # Create a DataFrame for the new row
        # Select relevant columns from df_exp for metadata, and then combine with new_metrics
        metadata_for_new_row = df_exp[['seed', 'run_id', 'model_name', 'type']].to_dict()
        for key, value in metadata_for_new_row.items():
            df_new_row[key] = value

        filepath = f"{results_dir}/logs/{experiment_name}/metrics_newth_{experiment_name}.csv"
        if os.path.exists(filepath):
            df_new_row.to_csv(filepath, mode="a", header=False, index=False, float_format='%.16g')
        else:
            df_new_row.to_csv(filepath, mode="w", header=True, index=False, float_format='%.16g')

        if verbose:
            print("-" * 60)

    np.savez(f"{results_dir}/logs/{experiment_name}/thresholds_{experiment_name}.npz", **all_optimal_thresholds)

    # Re-reading to get the full updated CSV content if it was saved
    if os.path.exists(filepath):
        return pd.read_csv(filepath)
    else:
        return pd.DataFrame([new_metrics]) # Fallback if file was never created (e.g., error in glob)

In [12]:
def change_threshold_to_check_metrics(model_class, model_config,
                     val_loader, experiment_name,
                     device='cpu',
                     results_dir="./results_earlystopping",
                     verbose=False):

    df_metrics = pd.read_csv(f"{results_dir}/logs/{experiment_name}/run_metrics_{experiment_name}.csv")

    all_optimal_thresholds = {}
    all_results_in_memory = []
    for exp_id in range(len(df_metrics)):
        # Setup
        df_exp = df_metrics.iloc[exp_id,:].copy()
        seed = df_exp['seed']

        model_config['model_name'] = df_exp['model_name']

        if verbose:
            print(f"\nChanging threshold for: {model_config['model_name']}")

        model_config['type'] = df_exp['type']
        run_id = df_exp['extra_run_id']

        pos_weight = torch.tensor([model_config['extra_params']['pos_weight']]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

        if verbose:
            print("\n OLD METRICS (MAX RECALL FOR PRECISION=0.9):")
            print(f" Precision: {df_exp['Precision']:.4f} | Recall: {df_exp['Recall']:.4f} | F1: {df_exp['F1']:.4f} | F2: {df_exp['F2']:.4f} | AUC-PR: {df_exp['AUC-PR']:.4f} | AUC-ROC: {df_exp['AUC-ROC']:.4f} | FPR: {df_exp['FPR']:.4f}")

        # Load model
        model_paths = glob.glob(os.path.join(results_dir,"saved_models",experiment_name,f"{run_id}_*.pth"))
        # Ensure we are loading a single file, assuming glob returns a list with one item
        model_path = model_paths[0] if model_paths else None
        if model_path is None:
            print(f"Error: No model found for run_id {run_id} in experiment {experiment_name}")
            continue

        model = model_class(**model_config['model_params']).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))

        # Evaluate
        model.eval()
        _, y_true, y_probs = evaluate(model, val_loader, criterion, device, is_temporal)

        # New threshold (max F1)
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
        precisions = precisions[:-1]
        recalls = recalls[:-1]

        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        new_best_th = thresholds[best_idx]
        all_optimal_thresholds[f"seed_{seed}"] = new_best_th

        # Calculate metrics with new threshold
        new_metrics = calculate_metrics_gnn(y_true, y_probs, prob_threshold=new_best_th)
        new_metrics['optimal_threshold'] = new_best_th

        if verbose:
            print("\n NEW METRICS (MAX F1):")
            print(f" Precision: {new_metrics['Precision']:.4f} | Recall: {new_metrics['Recall']:.4f} | F1: {new_metrics['F1']:.4f} | F2: {new_metrics['F2']:.4f} | AUC-PR: {new_metrics['AUC-PR']:.4f} | AUC-ROC: {new_metrics['AUC-ROC']:.4f} | FPR: {new_metrics['FPR']:.4f}\n")


        # Save metrics and metadata to a temporary DataFrame and append to list
        df_new_row = pd.DataFrame([new_metrics])
        metadata_for_new_row = df_exp[['seed', 'run_id', 'model_name', 'type']].to_dict()
        for key, value in metadata_for_new_row.items():
            df_new_row[key] = value
        all_results_in_memory.append(df_new_row)

        if verbose:
            print("-" * 60)

    # After the loop, concatenate all results into a single DataFrame
    if all_results_in_memory:
        final_df = pd.concat(all_results_in_memory, ignore_index=True)
        filepath = f"{results_dir}/logs/{experiment_name}/PRUEBAmetrics_newth_{experiment_name}.csv"
        # Write the final DataFrame to CSV once with full precision
        final_df.to_csv(filepath, mode="w", header=True, index=False, float_format='%.16g')
        return final_df
    else:
        return pd.DataFrame()

    #     # Save
    #     df_new_row = pd.DataFrame([new_metrics]) # Create a DataFrame for the new row
    #     # Select relevant columns from df_exp for metadata, and then combine with new_metrics
    #     metadata_for_new_row = df_exp[['seed', 'run_id', 'model_name', 'type']].to_dict()
    #     for key, value in metadata_for_new_row.items():
    #         df_new_row[key] = value

    #     filepath = f"{results_dir}/logs/{experiment_name}/metrics_newth_{experiment_name}.csv"
    #     if os.path.exists(filepath):
    #         df_new_row.to_csv(filepath, mode="a", header=False, index=False, float_format='%.16g')
    #     else:
    #         df_new_row.to_csv(filepath, mode="w", header=True, index=False, float_format='%.16g')

    #     print("-" * 60)

    # np.savez(f"{results_dir}/logs/{experiment_name}/thresholds_{experiment_name}.npz", **all_optimal_thresholds)

    # # Re-reading to get the full updated CSV content if it was saved
    # if os.path.exists(filepath):
    #     return pd.read_csv(filepath)
    # else:
    #     return pd.DataFrame([new_metrics]) # Fallback if file was never created (e.g., error in glob)

# Configuration

In [13]:
ROOT_PATH = "./dataset_processed"

In [14]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [15]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [16]:
model_config = {
    "model_name": None,
    "type": None,
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

# Main

In [17]:
RESULTS_DIR = "./results_earlystopping"

pairs = [(SimpleMLP, "SimpleMLP_BiasOn"),
          (EdgeGRU_Baseline, "EdgeGRU_BiasOn"),
          (StaticGNN_Identity, "StaticGNN_BiasOn_robust_Identity"),
          (ST_GNN_Identity, "ST_GNN_BiasOn_robust_Identity_clone")]

In [18]:
for i, (mclass, exp_name) in enumerate(pairs):
    _=change_threshold(mclass, model_config, val_loader, exp_name, DEVICE, RESULTS_DIR, True)
    if i < len(pairs) - 1: # Only print separators if it's not the last iteration
        print("="*70)
        print("\n")
        print("="*70)



Changing threshold for: SimpleMLP_BiasOn_seed42

 OLD METRICS (MAX RECALL FOR PRECISION=0.9):
 Precision: 0.9015 | Recall: 0.0190 | F1: 0.0372 | F2: 0.0236 | AUC-PR: 0.1836 | AUC-ROC: 0.7786 | FPR: 0.0001

 NEW METRICS (MAX F1):
 Precision: 0.4725 | Recall: 0.1531 | F1: 0.2313 | F2: 0.1771 | AUC-PR: 0.1836 | AUC-ROC: 0.7786 | FPR: 0.0071

------------------------------------------------------------

Changing threshold for: SimpleMLP_BiasOn_seed123

 OLD METRICS (MAX RECALL FOR PRECISION=0.9):
 Precision: 0.4528 | Recall: 0.1565 | F1: 0.2326 | F2: 0.1800 | AUC-PR: 0.1736 | AUC-ROC: 0.7757 | FPR: 0.0078

 NEW METRICS (MAX F1):
 Precision: 0.4528 | Recall: 0.1565 | F1: 0.2326 | F2: 0.1800 | AUC-PR: 0.1736 | AUC-ROC: 0.7757 | FPR: 0.0078

------------------------------------------------------------

Changing threshold for: SimpleMLP_BiasOn_seed777

 OLD METRICS (MAX RECALL FOR PRECISION=0.9):
 Precision: 0.9142 | Recall: 0.0435 | F1: 0.0831 | F2: 0.0537 | AUC-PR: 0.2251 | AUC-ROC: 0.8106 

# Main 2 (to analyze and check metrics)

In [19]:
MODEL_NAME_MAPPING = {
    'SimpleMLP_BiasOn': 'Simple MLP',
    'EdgeGRU_BiasOn': 'Edge GRU',
    'StaticGNN_BiasOn_robust_Identity': 'Static GNN',
    'ST_GNN_BiasOn_robust_Identity_clone': 'ST-GNN (Ours)'
}

all_results_dfs = []
for i, (mclass, exp_name) in enumerate(pairs):
    current_df = change_threshold_to_check_metrics(mclass, model_config, val_loader, exp_name, DEVICE, RESULTS_DIR, False)
    if not current_df.empty:
        all_results_dfs.append(current_df)

    # if i < len(pairs) - 1: # Only print separators if it's not the last iteration
    #     print("="*70)
    #     print("\n")
    #     print("="*70)

if all_results_dfs:
    final_combined_df = pd.concat(all_results_dfs, ignore_index=True)
    # Extract the base model name by splitting at '_seed' and then apply the mapping
    final_combined_df['model'] = final_combined_df['model_name'].apply(lambda x: MODEL_NAME_MAPPING.get(x.split('_seed')[0], x.split('_seed')[0]))
    print("\nAll evaluation results combined")
else:
    print("No evaluation results were generated.")


All evaluation results combined


In [20]:
def generate_summary_table(df):
    possible = ['Precision', 'Recall', 'F1', 'F2', 'AUC-PR', 'AUC-ROC', 'FPR', 'optimal_threshold']
    metrics = [c for c in df.columns if c in possible]
    model_order = ["Simple MLP", "Edge GRU", "Static GNN", "ST-GNN (Ours)"]

    # Convert 'model' column to categorical type with the desired order
    df['model'] = pd.Categorical(df['model'], categories=model_order, ordered=True)

    summary = df.groupby('model', observed=False)[metrics].agg(['mean', 'std'])

    print("\n" + "="*80)
    print(" SUMMARY TABLE (Mean ± Std)")
    print("="*80)
    print(summary.round(4))


generate_summary_table(final_combined_df)


 SUMMARY TABLE (Mean ± Std)
              Precision          Recall              F1              F2  \
                   mean     std    mean     std    mean     std    mean   
model                                                                     
Simple MLP       0.4992  0.0499  0.1614  0.0147  0.2439  0.0223  0.1867   
Edge GRU         0.2624  0.0727  0.3329  0.0779  0.2832  0.0457  0.3075   
Static GNN       0.5269  0.1673  0.3719  0.0556  0.4315  0.1003  0.3927   
ST-GNN (Ours)    0.7454  0.0546  0.7742  0.0609  0.7564  0.0183  0.7662   

                       AUC-PR         AUC-ROC             FPR          \
                  std    mean     std    mean     std    mean     std   
model                                                                   
Simple MLP     0.0170  0.1955  0.0244  0.7883  0.0186  0.0067  0.0008   
Edge GRU       0.0519  0.2327  0.0775  0.8328  0.0443  0.0429  0.0215   
Static GNN     0.0717  0.4024  0.1318  0.8670  0.0503  0.0164  0.0122   
ST-GNN 

In [22]:
def evaluate_test1(model_class, model_config,
                        test_loader,
                        experiment_name,
                        device,
                        results_gral_dir,
                        #results_test_dirname,
                        verbose=False
                   ):

    # os.makedirs(os.path.join(results_gral_dir, results_test_dirname), exist_ok=True)

    df_metrics = pd.read_csv(f"{results_gral_dir}/logs/{experiment_name}/metrics_newth_{experiment_name}.csv")
    opt_thresholds = np.load(f"{results_gral_dir}/logs/{experiment_name}/thresholds_{experiment_name}.npz")
    all_results_in_memory = []

    for exp_id in range(len(df_metrics)):
        # Setup
        df_exp = df_metrics.iloc[exp_id,:].copy()
        seed = df_exp['seed']
        opt_th = opt_thresholds[f"seed_{seed}"]

        raw_model_name = df_exp['model_name']

        model_config['model_name'] = raw_model_name

        if verbose:
            print(f"\nEvaluating: {raw_model_name}")

        model_config['type'] = df_exp['type']
        run_id = df_exp['run_id']

        pos_weight = torch.tensor([model_config['extra_params']['pos_weight']]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

        # Load model
        model_paths = glob.glob(os.path.join(results_gral_dir,"saved_models",experiment_name,f"{run_id}_*.pth"))
        model_path = model_paths[0] if model_paths else None
        if model_path is None:
            print(f"Error: No model found for run_id {run_id} in experiment {experiment_name}")
            continue

        model = model_class(**model_config['model_params']).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))

        # Evaluate in test
        model.eval()
        _, y_true, y_probs = evaluate(model, test_loader, criterion, device, is_temporal)

        metrics = calculate_metrics_gnn(y_true, y_probs, prob_threshold=opt_th)

        if verbose:
            print(f" Precision: {metrics['Precision']:.4f} | Recall: {metrics['Recall']:.4f} | F1: {metrics['F1']:.4f} | F2: {metrics['F2']:.4f} | AUC-PR: {metrics['AUC-PR']:.4f} | AUC-ROC: {metrics['AUC-ROC']:.4f} | FPR: {metrics['FPR']:.4f}\n")

        # Save metrics and metadata to a temporary DataFrame and append to list
        df_new_row = pd.DataFrame([metrics])
        metadata_for_new_row = df_exp[['optimal_threshold', 'seed', 'run_id', 'model_name', 'type']].to_dict()
        for key, value in metadata_for_new_row.items():
            df_new_row[key] = value
        all_results_in_memory.append(df_new_row)

        if verbose:
            print("-" * 60)

    # After the loop, concatenate all results into a single DataFrame
    if all_results_in_memory:
        final_df = pd.concat(all_results_in_memory, ignore_index=True)
        # filepath = f"{results_gral_dir}/{results_test_dirname}/test1_metrics_{experiment_name}.csv"
        # final_df.to_csv(filepath, mode="w", header=True, index=False, float_format='%.16g')
        return final_df
    else:
        return pd.DataFrame() # Return an empty DataFrame if no results were generated

In [23]:
BASE_RESULTS_PATH = "./results_earlystopping/"


all_results_dfs2 = []
for i, (mclass, exp_name) in enumerate(pairs):
    current_df2 = evaluate_test1(mclass, model_config, val_loader, exp_name, DEVICE, BASE_RESULTS_PATH, False)
    if not current_df2.empty:
        all_results_dfs2.append(current_df2)

    # if i < len(pairs) - 1: # Only print separators if it's not the last iteration
    #     print("="*70)
    #     print("\n")
    #     print("="*70)

if all_results_dfs2:
    final_combined_df2 = pd.concat(all_results_dfs2, ignore_index=True)
    # Extract the base model name by splitting at '_seed' and then apply the mapping
    final_combined_df2['model'] = final_combined_df2['model_name'].apply(lambda x: MODEL_NAME_MAPPING.get(x.split('_seed')[0], x.split('_seed')[0]))
    print("\nAll evaluation results combined")
    generate_summary_table(final_combined_df2)
else:
    print("No evaluation results were generated.")


All evaluation results combined

 SUMMARY TABLE (Mean ± Std)
              Precision          Recall              F1              F2  \
                   mean     std    mean     std    mean     std    mean   
model                                                                     
Simple MLP       0.4992  0.0499  0.1614  0.0147  0.2439  0.0223  0.1867   
Edge GRU         0.2624  0.0727  0.3329  0.0779  0.2832  0.0457  0.3075   
Static GNN       0.5269  0.1673  0.3719  0.0556  0.4315  0.1003  0.3927   
ST-GNN (Ours)    0.7454  0.0546  0.7742  0.0609  0.7564  0.0183  0.7662   

                       AUC-PR         AUC-ROC             FPR          \
                  std    mean     std    mean     std    mean     std   
model                                                                   
Simple MLP     0.0170  0.1955  0.0244  0.7883  0.0186  0.0067  0.0008   
Edge GRU       0.0519  0.2327  0.0775  0.8328  0.0443  0.0429  0.0215   
Static GNN     0.0717  0.4024  0.1318  0.8670  